# 📊 Evaluasi Kinerja Sistem Briefly

Notebook ini mengukur kualitas keempat *worker* AI pada pipeline berita. Jalankan sel **berurutan dari atas ke bawah**.

| Worker | Yang diukur | Metrik | Ground truth |
|--------|-------------|--------|--------------|
| **1 — Clustering** | Artikel sekelompok mirip & antar-klaster terpisah | Silhouette, Davies-Bouldin | — (*unsupervised*) |
| **2 — Summarization** | Kemiripan ringkasan model dgn ringkasan manusia | ROUGE-1/2/L + **BERTScore** | **XL-Sum** (BBC) & **IndoSum** (Liputan6) |
| **3 — Sentimen (utama)** | Akurasi klasifikasi Positif/Negatif/Netral | F1-macro, Akurasi | **IndoNLU SmSA** (label emas manusia) |
| **3 — Sentimen (aspek)** | Ketepatan sentimen terhadap aktor pada ringkasan | F1-macro + *agreement* | **Llama 3.3 70B** (LLM-as-Judge) |
| **4 — Prediksi dampak** | Sektor & tingkat risiko masuk akal | Akurasi | **Llama 3.3 70B** (LLM-as-Judge) |

**Dataset publik yang dipakai (diakses LANGSUNG, tanpa loading-script):**
- **XL-Sum** (`csebuetnlp/xlsum`, subset *indonesian*) — artikel + ringkasan jurnalis **BBC Indonesia**. Diunduh dari **parquet auto-convert** HuggingFace (branch `refs/convert/parquet`), jadi tak butuh `trust_remote_code` maupun `datasets<3.0.0`.
- **IndoNLU SmSA** (`smsa_doc-sentiment-prosa`) — sentimen berlabel emas (positif/netral/negatif), benchmark standar NLP Indonesia. Diunduh dari **TSV resmi** repo `IndoNLP/indonlu` di GitHub (pandas, tanpa library `datasets`).
- **IndoSum/Liputan6** (`id_liputan6`) — domain berita paling mirip Briefly. *Butuh unduh data manual (lisensi); otomatis di-skip bila `INDOSUM_DATA_DIR` kosong.*

**Optimasi runtime:** panggilan Groq dijalankan **paralel** (`ThreadPoolExecutor`, lihat `MAKS_PARALEL`) dengan helper `groq_chat` yang otomatis **retry + rotasi API key** saat kena rate limit — jauh lebih cepat & tahan-banting dibanding loop serial + `time.sleep`.

**Cara baca metrik:**
- **Silhouette** (−1…1): →1 = artikel pas di klasternya; 0 = di perbatasan; negatif = kemungkinan salah klaster. *(Dihitung pada embedding yang sama dengan algoritma, jadi cenderung optimistis — baca sebagai indikator overlap.)*
- **Davies-Bouldin** (≥0): makin kecil makin baik.
- **ROUGE** = tumpang-tindih **kata** (leksikal). **BERTScore** = kemiripan **makna** (semantik) — menangkap parafrasa ("koruptor" ≈ "korupsi") yang dilewatkan ROUGE.
- **F1-macro**: rata-rata seimbang precision & recall tiap kelas — tak bisa "curang" dengan menebak satu kelas saja.

**Catatan biaya:** evaluasi memanggil Groq ratusan kali (ROUGE 2 dataset + SmSA + LLM-judge). Atur `N_SUM` / `N_SENT` / `batas_klaster` untuk hemat kuota, dan `MAKS_PARALEL` untuk menyeimbangkan kecepatan vs rate limit.

In [1]:
# Akses dataset LANGSUNG via parquet (XL-Sum) + TSV (SmSA) -> tidak butuh loading-script,
# sehingga TIDAK perlu lagi pin "datasets<3.0.0" yang rapuh.
# python-dotenv: baca kredensial dari file .env (bukan hardcode -> tak gampang dicabut Groq).
!pip install supabase sentence-transformers scikit-learn pandas pyarrow rouge-score bert-score groq python-dotenv -q

In [2]:
# ============================================================
# SETUP — koneksi, model, helper (identik dgn pipeline produksi)
# ============================================================
import os
import re
import json
import time
import threading

import numpy as np
from sklearn.metrics import silhouette_score, davies_bouldin_score, f1_score
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer
from supabase import create_client
from groq import Groq
from dotenv import load_dotenv

# ----- Baca kredensial dari .env (bukan hardcode) -----
load_dotenv()  # mencari file .env di folder kerja
SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_SERVICE_KEY"]      # evaluasi butuh akses penuh (service_role)
GROQ_API_KEYS = [k.strip() for k in os.environ.get("GROQ_API_KEYS", "").split(",") if k.strip()]
if not GROQ_API_KEYS:
    raise RuntimeError("GROQ_API_KEYS kosong — pastikan file .env ada & terisi di folder ini.")

# Validasi cepat: peringatkan bila ada key dengan panjang tak wajar (cegah 401 senyap).
_panjang_aneh = [i for i, k in enumerate(GROQ_API_KEYS) if len(k) != 56]
if _panjang_aneh:
    print(f"⚠️  Key indeks {_panjang_aneh} panjangnya bukan 56 char -> kemungkinan tidak sah (401)!")

# ----- Supabase -----
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ----- Model embedding (sama persis dengan Worker 3) -----
print("Memuat model embedding...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# ----- Groq (model pekerja + juri LLM-as-Judge) -----
# CATATAN: ini BUKAN model produksi (llama-3.1-8b) -> kita uji model lain di sini.
# Juri (gpt-oss) = model reasoning -> output bisa menyertakan <think> (dibersihkan di groq_chat).
MODEL_PEKERJA = "llama-3.1-8b-instant"      # model yang dievaluasi (sama dengan produksi)
MODEL_JURI = "openai/gpt-oss-120b"          # juri LLM-as-Judge (lebih besar -> menilai yang kecil)

# Satu client per key + rotasi thread-safe. Memungkinkan evaluasi PARALEL (ThreadPool)
# tanpa balapan state global, dan otomatis pindah key saat salah satu kena rate limit / 401.
_groq_clients = [Groq(api_key=k) for k in GROQ_API_KEYS]
_key_lock = threading.Lock()
_key_idx = 0

# Pembersih jejak reasoning: model reasoning kerap menyisipkan <think>...</think> di konten,
# yang akan merusak ROUGE/BERTScore (summarization) & json.loads (sentimen/juri).
_RE_THINK = re.compile(r"<think>.*?</think>", re.DOTALL)


def groq_chat(messages, model, **kwargs):
    """Panggil Groq dgn retry + rotasi key otomatis. Aman dipakai banyak thread.
    Memutar ke key lain saat rate limit (429/5xx) ATAU key tidak sah (401) -> satu key buruk
    tidak meracuni semua panggilan. Konten dibersihkan dari blok <think> (model reasoning).
    Melempar exception bila SEMUA key gagal."""
    global _key_idx
    last_err = None
    for _ in range(2 * len(_groq_clients)):
        with _key_lock:
            idx = _key_idx
        try:
            resp = _groq_clients[idx].chat.completions.create(
                messages=messages, model=model, **kwargs)
            isi = resp.choices[0].message.content or ""
            return _RE_THINK.sub("", isi).strip()      # buang reasoning <think>...</think>
        except Exception as e:
            last_err = e
            pesan = str(e).lower()
            transient = any(t in pesan for t in ("rate", "limit", "429", "503", "timeout", "overload"))
            auth = any(t in pesan for t in ("401", "invalid api key", "invalid_api_key", "unauthorized"))
            if transient or auth:
                with _key_lock:
                    if _key_idx == idx:               # pindah ke key berikutnya (sekali per kegagalan)
                        _key_idx = (_key_idx + 1) % len(_groq_clients)
                if transient:
                    time.sleep(2)
            else:
                raise
    raise last_err


def _ekstrak_json(isi):
    """Ambil objek JSON dari balasan model TANPA bergantung pada mode json_object Groq
    (yang sering gagal divalidasi pada model reasoning -> error 400).
    Coba json.loads langsung; bila gagal, ambil blok {...} pertama-hingga-terakhir."""
    if isinstance(isi, (dict, list)):
        return isi
    s = str(isi).strip()
    try:
        return json.loads(s)
    except Exception:
        m = re.search(r"\{.*\}", s, re.DOTALL)
        if m:
            return json.loads(m.group(0))
        raise ValueError(f"tidak ada JSON valid di balasan: {s[:120]}")


# ----- Helper identik dengan pipeline produksi (agar evaluasi setara) -----
def bersihkan_teks_struktural(teks):
    return re.sub(r"\s+", " ", str(teks)).strip()


# WAJIB sinkron dengan knob Worker 3 (notebook pipeline). Nilai data-driven hasil tuner.
BOBOT_JUDUL = 0.45


def vektorisasi_berbobot(judul, isi):
    """Vektor gabungan judul (BOBOT_JUDUL) + isi (1-BOBOT_JUDUL), L2 — SAMA PERSIS dengan Worker 3."""
    teks_judul = bersihkan_teks_struktural(judul)
    teks_isi = bersihkan_teks_struktural(" ".join(str(isi).split()[:150]))
    if teks_isi:
        vec_judul, vec_isi = embed_model.encode([teks_judul, teks_isi])
    else:
        vec_judul = embed_model.encode([teks_judul])[0]
        vec_isi = vec_judul
    vektor_kombinasi = (vec_judul * BOBOT_JUDUL) + (vec_isi * (1.0 - BOBOT_JUDUL))
    return vektor_kombinasi / np.linalg.norm(vektor_kombinasi)


def pisah_kalimat(teks):
    """Pemisah kalimat ringan untuk Bahasa Indonesia (tanpa dependensi nltk/punkt)."""
    kalimat = re.split(r"(?<=[.!?])\s+", str(teks).strip())
    return [k for k in kalimat if k]


def _normalisasi_sentimen(nilai):
    """Seragamkan label sentimen ke BINER Positif/Negatif (default Negatif).
    Apa pun yang tidak diawali 'pos' (termasuk 'Netral') dianggap Negatif."""
    return "Positif" if str(nilai).strip().lower().startswith("pos") else "Negatif"


# Wadah hasil semua evaluasi (dipakai cetak_laporan di sel terakhir)
HASIL = {}
print(f"Setup selesai. ({len(GROQ_API_KEYS)} Groq key | pekerja={MODEL_PEKERJA} | juri={MODEL_JURI})")

c:\Users\dwiwi\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\dwiwi\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Memuat model embedding...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Setup selesai. (4 Groq key | pekerja=llama-3.1-8b-instant | juri=openai/gpt-oss-120b)


In [3]:
# ============================================================
# DIAGNOSTIK — cek kesehatan tiap Groq API key (jalankan bila banyak 401/429)
#   Kirim 1 permintaan SANGAT kecil per key -> tahu mana OK / invalid / rate-limit.
#   AMAN: TIDAK pernah menampilkan API key (hanya 4 char terakhir + status + indeks).
# ============================================================
def cek_kesehatan_key(model=MODEL_PEKERJA):
    print(f'Cek {len(GROQ_API_KEYS)} key terhadap model {model}...')
    print()
    invalid = []
    for i, k in enumerate(GROQ_API_KEYS):
        try:
            Groq(api_key=k).chat.completions.create(
                model=model, messages=[{'role': 'user', 'content': 'hi'}], max_tokens=1)
            status = 'OK (valid)'
        except Exception as e:
            m = str(e).lower()
            if '401' in m or 'invalid api key' in m or 'unauthorized' in m:
                status = 'INVALID (401)'
                invalid.append(i)
            elif '429' in m or 'rate limit' in m:
                status = 'valid tapi rate-limited (429)'
            elif 'model' in m and any(t in m for t in ('not found', 'does not exist', 'decommission')):
                status = 'MODEL TIDAK ADA'
                invalid.append(i)
            else:
                status = f'ERROR: {str(e)[:50]}'
                invalid.append(i)
        # hanya 4 char terakhir, BUKAN key penuh
        print(f'  key[{i}] (...{k[-4:]})  -> {status}')
        time.sleep(0.4)
    sehat = len(GROQ_API_KEYS) - len(invalid)
    print()
    print(f'Valid: {sehat}/{len(GROQ_API_KEYS)}')
    if invalid:
        print(f'Hapus key pada INDEKS {invalid} dari GROQ_API_KEYS di .env (key-nya tidak ditampilkan).')
    else:
        print('Semua key valid.')
    return None   # tidak mengembalikan daftar key -> tak bocor ke output notebook


cek_kesehatan_key()

Cek 4 key terhadap model llama-3.1-8b-instant...

  key[0] (...Ulkr)  -> OK (valid)
  key[1] (...17hS)  -> OK (valid)
  key[2] (...Udas)  -> OK (valid)
  key[3] (...WNjw)  -> OK (valid)

Valid: 4/4
Semua key valid.


In [4]:
# ============================================================
# DATASET REFERENSI PUBLIK — akses LANGSUNG tanpa loading-script
#   XL-Sum (BBC)   -> parquet auto-convert HuggingFace (branch refs/convert/parquet)
#   IndoNLU SmSA   -> TSV resmi repo indobenchmark/indonlu (GitHub)
#   IndoSum        -> file lokal hasil ekstrak indosum.tar.gz (Kurniawan & Louvan 2018)
# Keunggulan: tanpa trust_remote_code & tanpa pin datasets<3.0.0 (jauh lebih stabil).
# ============================================================
import io
import os
import time
import urllib.request
from collections import Counter

import pandas as pd

# ── PARAMETER JUMLAH DATA (knob utama) ──
# Makin besar = metrik makin STABIL/terpercaya, TAPI: (a) bukan bikin model lebih bagus,
# (b) makin banyak panggilan Groq (lebih lama & boros kuota), (c) DIBATASI ukuran test set.
# Kapasitas test set (maksimum yang masuk akal):
#   XL-Sum indonesian  : ~1.100  | IndoSum test.01 : 3.762  | SmSA biner (pos+neg) : ~370
# Minta lebih dari yang tersedia -> otomatis dipotong (min) ke jumlah yang ada.
N_SUM = 200              # sampel per dataset summarization (XL-Sum & IndoSum)
N_SENT = 350             # sampel sentimen SmSA biner (mendekati maksimum ~370)
SEED = 50                # acak reproducible (WAJIB: test set sering terurut per label)

# Folder hasil ekstrak indosum.tar.gz (berisi test.01.jsonl dst). None = lewati IndoSum.
INDOSUM_DIR = r"indosum/indosum"

_HDR = {"User-Agent": "Mozilla/5.0"}


class _LewatiDataset(Exception):
    """Penanda dataset sengaja dilewati (bukan error/jaringan)."""


def _unduh(url, timeout=120):
    """Unduh bytes mentah dari URL (HF/GitHub) dengan timeout longgar."""
    req = urllib.request.Request(url, headers=_HDR)
    return urllib.request.urlopen(req, timeout=timeout).read()


def _muat(nama, fn, maks_coba=3):
    """Jalankan loader; retry untuk gangguan jaringan, skip rapi bila sengaja dilewati/gagal."""
    for percobaan in range(1, maks_coba + 1):
        try:
            data = fn()
            print(f"  [OK]   {nama}: {len(data)} sampel")
            return data
        except _LewatiDataset as e:
            print(f"  [SKIP] {nama}: {e}")
            return []
        except Exception as e:
            pesan = str(e)[:140]
            if percobaan < maks_coba:
                print(f"  [coba lagi {percobaan}/{maks_coba - 1}] {nama}: {pesan}")
                time.sleep(3)
            else:
                print(f"  [SKIP] {nama}: {pesan}")
                return []
    return []


def _ambil_n(df, n):
    """Acak reproducible lalu ambil min(n, tersedia); cetak 'dipakai X dari Y tersedia'."""
    tersedia = len(df)
    dipakai = min(n, tersedia)
    print(f"         dipakai {dipakai} dari {tersedia} tersedia"
          + ("  (sudah maksimum test set)" if dipakai == tersedia else ""))
    return df.sample(n=dipakai, random_state=SEED)


# ── XL-Sum (BBC Indonesia) ── parquet auto-convert: kolom id/url/title/summary/text
XLSUM_URL = ("https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/"
             "refs%2Fconvert%2Fparquet/indonesian/test/0000.parquet")


def _load_xlsum():
    df = _ambil_n(pd.read_parquet(io.BytesIO(_unduh(XLSUM_URL))), N_SUM)
    return [{"teks": r["text"], "ringkasan": r["summary"]} for _, r in df.iterrows()]


# ── IndoNLU SmSA ── TSV resmi (tanpa header): kolom 0=teks, 1=label(positive/neutral/negative)
# Sentimen sistem = BINER (Positif/Negatif). Sampel gold "neutral" DIBUANG agar benchmark adil.
SMSA_URL = ("https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/"
            "smsa_doc-sentiment-prosa/test_preprocess.tsv")
PETA_SMSA = {"positive": "Positif", "negative": "Negatif"}


def _load_smsa():
    df = pd.read_csv(io.BytesIO(_unduh(SMSA_URL)), sep="\t", header=None,
                     names=["teks", "label"], quoting=3)  # quoting=3=QUOTE_NONE
    df["label"] = df["label"].astype(str).str.strip().str.lower()
    df = df[df["label"].isin(("positive", "negative"))]   # buang netral (biner)
    df = _ambil_n(df, N_SENT)
    data = [{"teks": r["teks"], "emas": PETA_SMSA[r["label"]]} for _, r in df.iterrows()]
    # Tampilkan distribusi kelas agar ketahuan bila sampel timpang.
    print(f"         distribusi kelas: {dict(Counter(d['emas'] for d in data))}")
    return data


# ── IndoSum ── jsonl tokenized: paragraphs=list[par][kalimat][token], summary=list[kalimat-str]
def _gabung_paragraf(paras):
    return " ".join(" ".join(tok for tok in sent) for par in paras for sent in par)


def _gabung_ringkasan(summ):
    return " ".join(s if isinstance(s, str) else " ".join(s) for s in summ)


def _load_indosum():
    if not INDOSUM_DIR:
        raise _LewatiDataset("set INDOSUM_DIR ke folder hasil ekstrak indosum.tar.gz")
    path = os.path.join(INDOSUM_DIR, "test.01.jsonl")  # pakai 1 fold test (5 fold = data sama)
    if not os.path.exists(path):
        raise _LewatiDataset(f"file tidak ditemukan: {path}")
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            d = json.loads(line)
            rows.append({"teks": _gabung_paragraf(d["paragraphs"]),
                         "ringkasan": _gabung_ringkasan(d["summary"])})
    return _ambil_n(pd.DataFrame(rows), N_SUM).to_dict("records")


print("Memuat dataset referensi...")
SAMPEL_XLSUM = _muat("XL-Sum (BBC, summarization)", _load_xlsum)
SAMPEL_INDOSUM = _muat("IndoSum (summarization, domain mirip Briefly)", _load_indosum)
SAMPEL_SMSA = _muat("IndoNLU SmSA (sentimen, label emas, BINER)", _load_smsa)

Memuat dataset referensi...
         dipakai 200 dari 4780 tersedia
  [OK]   XL-Sum (BBC, summarization): 200 sampel
         dipakai 200 dari 3762 tersedia
  [OK]   IndoSum (summarization, domain mirip Briefly): 200 sampel
         dipakai 350 dari 412 tersedia
         distribusi kelas: {'Positif': 175, 'Negatif': 175}
  [OK]   IndoNLU SmSA (sentimen, label emas, BINER): 350 sampel


In [5]:
# ============================================================
# WORKER 1 — CLUSTERING (Silhouette & Davies-Bouldin)
#   Diukur HANYA pada klaster MATANG yang benar-benar dipakai produk:
#   status_summary=1 & status_prediksi=1 & jumlah_berita >= MIN_BERITA_KLASTER.
#   Silhouette butuh VEKTOR ARTIKEL -> ambil id klaster matang dari tabel_cluster,
#   lalu tarik artikel-artikelnya dari tabel_berita (tabel_cluster tak punya embedding).
# ============================================================
MIN_BERITA_KLASTER = 5


def eval_clustering(min_berita=MIN_BERITA_KLASTER, maks_artikel=2000):
    print("\n" + "=" * 55)
    print("EVALUASI WORKER 1 — CLUSTERING (klaster matang)")
    print("=" * 55)

    # 1) id klaster MATANG (dipakai end-to-end oleh produk).
    res_cl = (supabase.table("tabel_cluster")
              .select("id_cluster")
              .eq("status_summary", 1)
              .eq("status_prediksi", 1)
              .gte("jumlah_berita", min_berita)
              .execute())
    id_klaster = [c["id_cluster"] for c in res_cl.data]
    if len(id_klaster) < 2:
        print(f"Klaster matang (status=1 & jumlah_berita>={min_berita}) < 2 — "
              "jalankan Worker 3/4/5 dulu atau turunkan min_berita.")
        return None

    # 2) Tarik artikel milik klaster-klaster itu (silhouette butuh vektor per-artikel).
    res = (supabase.table("tabel_berita")
           .select("judul, isi_teks, id_cluster")
           .in_("id_cluster", id_klaster)
           .limit(maks_artikel).execute())
    if len(res.data) < 20:
        print("Artikel pada klaster matang < 20 — sampel terlalu kecil untuk metrik stabil.")
        return None

    print(f"Klaster matang : {len(id_klaster)}  |  artikel diuji: {len(res.data)}")
    print("Menghitung ulang vektor (vektorisasi_berbobot, sama dengan produksi)...")
    vektor = [vektorisasi_berbobot(b["judul"], b["isi_teks"]) for b in res.data]
    X = np.array(vektor)
    label = np.array([str(b["id_cluster"]) for b in res.data])

    # Buang singleton (anggota < 2) bila muncul akibat batas maks_artikel.
    nilai_unik, jumlah = np.unique(label, return_counts=True)
    klaster_valid = set(nilai_unik[jumlah >= 2])
    mask = np.array([l in klaster_valid for l in label])
    n_singleton = int((~mask).sum())
    X_valid = X[mask]
    y_valid = LabelEncoder().fit_transform(label[mask])
    n_klaster = len(set(y_valid))

    if n_klaster < 2 or len(X_valid) <= n_klaster:
        print("Klaster valid (>=2 anggota) kurang dari 2 — metrik tidak dapat dihitung.")
        return None

    sil = silhouette_score(X_valid, y_valid, metric="cosine")
    db = davies_bouldin_score(X_valid, y_valid)

    status_sil = "BAGUS" if sil > 0.5 else ("CUKUP" if sil > 0.25 else "PERLU REVIEW")
    status_db = "BAGUS" if db < 0.5 else ("CUKUP" if db < 1.0 else "PERLU REVIEW")

    print(f"\nArtikel diuji   : {len(X)}  (singleton dibuang: {n_singleton})")
    print(f"Klaster valid   : {n_klaster}")
    print("-" * 40)
    print(f"Silhouette Score : {sil:.4f}  -> {status_sil}  (target > 0.25, ideal > 0.5)")
    print(f"Davies-Bouldin   : {db:.4f}  -> {status_db}  (target < 1.0, ideal < 0.5)")
    print("\nCara baca:")
    print(f"  Silhouette {sil:.2f}: rata-rata artikel "
          f"{'sudah pas di klasternya' if sil > 0.25 else 'banyak berada di perbatasan'}")
    print(f"  DB {db:.2f}: antar-klaster {'cukup terpisah' if db < 1.0 else 'masih banyak overlap'}")
    print("\nCatatan: hanya klaster matang (dipakai produk) yang diukur — angka ini menilai")
    print("kualitas klaster yang DIKIRIM, bukan seluruh ekor klaster kecil/singleton.")

    return {"silhouette": round(float(sil), 4), "davies_bouldin": round(float(db), 4),
            "n_klaster": n_klaster, "n_singleton": n_singleton}


HASIL["clustering"] = eval_clustering()


EVALUASI WORKER 1 — CLUSTERING (klaster matang)
Klaster matang : 129  |  artikel diuji: 868
Menghitung ulang vektor (vektorisasi_berbobot, sama dengan produksi)...

Artikel diuji   : 868  (singleton dibuang: 0)
Klaster valid   : 129
----------------------------------------
Silhouette Score : 0.2880  -> CUKUP  (target > 0.25, ideal > 0.5)
Davies-Bouldin   : 1.7749  -> PERLU REVIEW  (target < 1.0, ideal < 0.5)

Cara baca:
  Silhouette 0.29: rata-rata artikel sudah pas di klasternya
  DB 1.77: antar-klaster masih banyak overlap

Catatan: hanya klaster matang (dipakai produk) yang diukur — angka ini menilai
kualitas klaster yang DIKIRIM, bukan seluruh ekor klaster kecil/singleton.


In [6]:
# ============================================================
# DIAGNOSTIK DAVIES-BOULDIN — bongkar DB per-klaster + lihat headroom
#   DB = rata-rata R_i ; R_i = max_j (s_i + s_j) / d(c_i, c_j).
#   Klaster dengan R_i TERTINGGI = penyetir DB (longgar / overlap dgn tetangga).
#   Dipakai untuk memutuskan: pangkas/pisah klaster jelek vs naikkan THRESHOLD.
# ============================================================
from sklearn.decomposition import PCA


def _db_per_cluster(X, y):
    """Kembalikan (db, rows) dengan rows = [(label, R_i, sebaran_i, tetangga)]."""
    labels = np.unique(y)
    cent = np.array([X[y == c].mean(axis=0) for c in labels])
    s = np.array([np.linalg.norm(X[y == c] - cent[i], axis=1).mean()
                  for i, c in enumerate(labels)])
    rows = []
    for i in range(len(labels)):
        best_r, best_j = -1.0, -1
        for j in range(len(labels)):
            if i == j:
                continue
            d = np.linalg.norm(cent[i] - cent[j])
            r = (s[i] + s[j]) / d if d > 0 else np.inf
            if r > best_r:
                best_r, best_j = r, j
        rows.append((labels[i], best_r, s[i], labels[best_j]))
    db = float(np.mean([r[1] for r in rows]))
    return db, sorted(rows, key=lambda r: r[1], reverse=True)


def diagnosa_db(min_berita=MIN_BERITA_KLASTER, maks_artikel=2000, n_pca=50, trim=0.10):
    res_cl = (supabase.table("tabel_cluster").select("id_cluster")
              .eq("status_summary", 1).eq("status_prediksi", 1)
              .gte("jumlah_berita", min_berita).execute())
    id_klaster = [c["id_cluster"] for c in res_cl.data]
    res = (supabase.table("tabel_berita").select("judul, isi_teks, id_cluster")
           .in_("id_cluster", id_klaster).limit(maks_artikel).execute())
    X = np.array([vektorisasi_berbobot(b["judul"], b["isi_teks"]) for b in res.data])
    label = np.array([str(b["id_cluster"]) for b in res.data])

    uniq, cnt = np.unique(label, return_counts=True)
    valid = set(uniq[cnt >= 2])
    mask = np.array([l in valid for l in label])
    X, label = X[mask], label[mask]

    db, rows = _db_per_cluster(X, label)
    print(f"Davies-Bouldin (penuh, {X.shape[1]}-dim) : {db:.4f}  |  {len(set(label))} klaster, {len(X)} artikel")
    print("-" * 66)
    print(f"{'id_cluster':>12} {'R_i':>7} {'sebaran':>8} {'overlap_dgn':>12}  ukuran")
    for lab, r, s_i, nb in rows[:10]:
        sz = int((label == lab).sum())
        print(f"{lab:>12} {r:>7.3f} {s_i:>8.3f} {nb:>12}  {sz}")

    # (a) Headroom METODOLOGIS: DB Euclidean menghukum dimensi tinggi.
    #     PCA tidak mengubah klaster, hanya cara mengukur -> angka lebih jujur.
    k = min(n_pca, X.shape[1], X.shape[0] - 1)
    Xp = PCA(n_components=k, random_state=0).fit_transform(X)
    db_pca, _ = _db_per_cluster(Xp, label)
    print("-" * 66)
    print(f"DB setelah PCA -> {k} dim              : {db_pca:.4f}")

    # (b) Headroom KERAPATAN: DB bila tiap klaster dipangkas {trim:.0%} anggota
    #     terjauh dari centroid = simulasi efek THRESHOLD_ATAS produksi lebih ketat.
    keep = np.ones(len(label), bool)
    for c in np.unique(label):
        idx = np.where(label == c)[0]
        cen = X[idx].mean(axis=0)
        d = np.linalg.norm(X[idx] - cen, axis=1)
        n_buang = int(len(idx) * trim)
        if n_buang:
            keep[idx[np.argsort(d)[-n_buang:]]] = False
    db_trim, _ = _db_per_cluster(X[keep], label[keep])
    print(f"DB bila pangkas {trim:.0%} terjauh/klaster  : {db_trim:.4f}  "
          f"(perkiraan gain bila THRESHOLD_ATAS dinaikkan)")
    return {"db_full": db, "db_pca": db_pca, "db_trim": db_trim, "rows": rows}


DIAG_DB = diagnosa_db()


Davies-Bouldin (penuh, 384-dim) : 1.7749  |  129 klaster, 868 artikel
------------------------------------------------------------------
  id_cluster     R_i  sebaran  overlap_dgn  ukuran
        3733   3.844    0.559         5556  17
        5556   3.844    0.520         3733  13
        4833   3.691    0.261         4981  5
        4981   3.691    0.179         4833  5
        4745   3.606    0.457         3733  8
        4711   3.252    0.410         5740  6
        5740   3.252    0.447         4711  7
        4555   2.913    0.525         4876  11
        4876   2.913    0.482         4555  7
        3726   2.881    0.555         5022  5
------------------------------------------------------------------
DB setelah PCA -> 50 dim              : 1.6509
DB bila pangkas 10% terjauh/klaster  : 1.7544  (perkiraan gain bila THRESHOLD_ATAS dinaikkan)


In [7]:
# ============================================================
# TUNING CLUSTERING (Worker 1) — cari BOBOT_JUDUL & THRESHOLD_ATAS OPTIMAL secara DATA-DRIVEN
#   REPLIKA PENUH Worker 3: vektor berbobot + threshold-atas + ZONA ABU-ABU NER + running-centroid.
#   (Versi lama hanya pakai 1 threshold tanpa zona NER -> kurang setara produksi; kini disetarakan.)
#   Worker 3 = online berbasis-threshold (bukan KMeans), jadi BUKAN elbow-k.
#   Diproses KRONOLOGIS (oldest->newest) meniru arus berita nyata.
#   Tujuan = MAKSIMALKAN (silhouette x coverage), BUKAN silhouette saja:
#   silhouette saja gampang tertipu over-segmentasi (banyak singleton -> silhouette palsu tinggi).
#   Dijalankan di BEBERAPA jendela waktu -> ambil parameter yang skornya paling STABIL lintas-window.
# ============================================================
from collections import defaultdict
from datetime import datetime, timedelta

# --- konstanta replika Worker 3 (produksi) ---
NER_MIN = 0.30            # ambang overlap entitas di zona abu-abu (sama dgn produksi)
BAWAH_OFFSET = 0.12       # produksi: THRESHOLD_BAWAH = THRESHOLD_ATAS - 0.12
STOP_WORDS_ENTITAS = frozenset({
    'di', 'ke', 'dari', 'pada', 'dalam', 'yaitu', 'untuk', 'yang', 'dan', 'ini', 'itu',
    'telah', 'sebuah', 'puluhan', 'ratusan', 'insiden', 'skuad', 'pasukan', 'tim',
    'berkat', 'setelah', 'menyusul', 'pecahan', 'sejumlah', 'kepala', 'beberapa',
    'banyak', 'surat', 'para', 'warga', 'calon', 'getaran', 'bencana', 'masyarakat',
    'korban', 'aksi', 'gelar', 'pawai', 'ri', 'the', 'akan', 'istana', 'presiden',
})


def _ekstrak_entitas(teks):
    ent = re.findall(r"\b[A-Z][a-zA-Z0-9]*\b", str(teks))
    return {e.lower() for e in ent if e.lower() not in STOP_WORDS_ENTITAS}


def _overlap(set_a, set_k):
    return len(set_a & set_k) / len(set_a) if set_a else 0.0


def _ambil_pool_artikel(limit=1200, hari=7):
    """Ambil artikel jendela `hari` terakhir, KRONOLOGIS (oldest->newest) seperti arus nyata."""
    batas = (datetime.now() - timedelta(days=hari)).strftime("%Y-%m-%d %H:%M:%S")
    res = (supabase.table("tabel_berita")
           .select("judul, isi_teks, waktu_rilis")
           .gte("waktu_rilis", batas)
           .order("waktu_rilis", desc=False)
           .limit(limit).execute())
    arts = [a for a in res.data if a.get("judul")]
    print(f"  jendela {hari} hari -> {len(arts)} artikel (batas >= {batas})")
    return arts


def _encode_judul_isi(arts):
    """Encode judul & 150-kata pertama isi SEKALI + ekstraksi entitas per artikel."""
    judul = [bersihkan_teks_struktural(a["judul"]) for a in arts]
    isi = [bersihkan_teks_struktural(" ".join(str(a["isi_teks"]).split()[:150])) for a in arts]
    Vj = np.asarray(embed_model.encode(judul, batch_size=64, show_progress_bar=False))
    Vi = np.asarray(embed_model.encode(isi, batch_size=64, show_progress_bar=False))
    ent = [_ekstrak_entitas(a["isi_teks"]) for a in arts]
    return Vj, Vi, ent


def _vektor_berbobot_batch(Vj, Vi, bobot):
    V = Vj * bobot + Vi * (1.0 - bobot)
    norms = np.linalg.norm(V, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return V / norms


def _cluster_worker3(X, ent, thr_atas, thr_bawah):
    """Replika PERSIS penugasan Worker 3: atas->gabung; [bawah,atas)->gabung bila NER overlap>=NER_MIN."""
    cent, cnt, ents = [], [], []
    labels = np.empty(len(X), dtype=int)
    for i, v in enumerate(X):
        target = None
        if cent:
            sims = np.asarray(cent) @ v
            j = int(np.argmax(sims))
            if sims[j] >= thr_atas:
                target = j
            elif sims[j] >= thr_bawah and _overlap(ent[i], ents[j]) >= NER_MIN:
                target = j
        if target is not None:
            labels[i] = target
            cnt[target] += 1
            c = cent[target] + (v - cent[target]) / cnt[target]
            cent[target] = c / np.linalg.norm(c)
            ents[target] |= ent[i]
        else:
            labels[i] = len(cent)
            cent.append(v.copy()); cnt.append(1); ents.append(set(ent[i]))
    return labels


def _skor(X, labels):
    """Silhouette + Davies-Bouldin (buang singleton). NaN bila tak terdefinisi."""
    u, cnt = np.unique(labels, return_counts=True)
    valid = set(u[cnt >= 2])
    mask = np.array([l in valid for l in labels])
    n_clu, n_sing = len(set(labels)), int((~mask).sum())
    if mask.sum() <= len(set(labels[mask])) or len(set(labels[mask])) < 2:
        return float("nan"), float("nan"), n_clu, n_sing
    y = LabelEncoder().fit_transform(labels[mask])
    sil = silhouette_score(X[mask], y, metric="cosine")
    db = davies_bouldin_score(X[mask], y)
    return sil, db, n_clu, n_sing


# Grid DIPERLUAS agar mencakup wilayah threshold produksi (0.72+) yang dulu tak teruji.
BOBOT_GRID = (0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70)
ATAS_GRID = (0.55, 0.60, 0.65, 0.70, 0.72, 0.75, 0.78, 0.80)


def tuning_clustering_window(hari, limit=1200):
    """Sweep bobot x threshold-atas pada satu jendela waktu (THRESHOLD_BAWAH=atas-0.12)."""
    arts = _ambil_pool_artikel(limit, hari)
    print(f"\n{'='*72}\nWINDOW {hari} HARI | {len(arts)} artikel\n{'='*72}")
    if len(arts) < 30:
        print("  Pool < 30 artikel — perbesar `hari`/`limit` atau tunggu data.")
        return []
    Vj, Vi, ent = _encode_judul_isi(arts)
    n = len(arts)
    print(f"{'bobot':>6}{'atas':>6}{'bawah':>6}{'#klu':>6}{'sing%':>7}{'cover%':>8}{'silhou':>9}{'davies':>9}{'skor':>8}")
    hasil = []
    for b in BOBOT_GRID:
        X = _vektor_berbobot_batch(Vj, Vi, b)
        for ta in ATAS_GRID:
            tb = round(ta - BAWAH_OFFSET, 2)
            lab = _cluster_worker3(X, ent, ta, tb)
            sil, db, nclu, nsing = _skor(X, lab)
            cov = 1.0 - nsing / n
            sk = sil * cov if sil == sil else float("nan")
            hasil.append((hari, b, ta, tb, nclu, nsing, cov, sil, db, sk))
            f = lambda x: f"{x:.3f}" if x == x else "  nan"
            print(f"{b:6.2f}{ta:6.2f}{tb:6.2f}{nclu:6d}{nsing/n:7.0%}{cov:8.0%}{f(sil):>9}{f(db):>9}{f(sk):>8}")
    return hasil


def tuning_clustering(windows=(1, 3, 7), limit=1200):
    """Tuning lintas-jendela -> rekomendasikan parameter dgn skor RATA-RATA tertinggi & STABIL."""
    semua = []
    for h in windows:
        semua += tuning_clustering_window(h, limit)
    valid = [r for r in semua if r[9] == r[9]]
    if not valid:
        print("\nSemua kombinasi NaN — topik nyaris tak berulang di jendela ini.")
        return semua
    print(f"\n{'#'*72}\nOPTIMAL PER WINDOW (maksimalkan silhouette x coverage)\n{'#'*72}")
    for h in windows:
        sub = [r for r in valid if r[0] == h]
        if sub:
            best = max(sub, key=lambda r: r[9])
            print(f" {h} hari -> BOBOT={best[1]:.2f} ATAS={best[2]:.2f} BAWAH={best[3]:.2f} "
                  f"| skor={best[9]:.4f} sil={best[7]:.4f} cover={best[6]:.0%} DB={best[8]:.4f} #klu={best[4]}")
    agg = defaultdict(list)
    for r in valid:
        agg[(r[1], r[2])].append(r[9])
    ranked = sorted(((np.mean(v), len(v), k) for k, v in agg.items() if len(v) >= max(2, len(windows) - 1)),
                    reverse=True)
    print(f"\n{'='*72}\nPALING STABIL (rata-rata skor lintas-window)\n{'='*72}")
    for mean_sk, cnt, (b, ta) in ranked[:8]:
        print(f"  BOBOT={b:.2f} ATAS={ta:.2f} BAWAH={ta - BAWAH_OFFSET:.2f} | skor_rata2={mean_sk:.4f} (di {cnt} window)")
    if ranked:
        mean_sk, cnt, (b, ta) = ranked[0]
        print(f"\n>> REKOMENDASI FINAL (tempel ke knob Worker 3 pipeline utama):")
        print(f"     BOBOT_JUDUL = {b:.2f}")
        print(f"     THRESHOLD_ATAS = {ta:.2f}")
        print(f"     THRESHOLD_BAWAH = {ta - BAWAH_OFFSET:.2f}")
        print(f"     NER_MIN = {NER_MIN}")
    return semua


HASIL_TUNING = tuning_clustering(windows=(1, 3, 7))


  jendela 1 hari -> 1000 artikel (batas >= 2026-06-18 19:55:23)

WINDOW 1 HARI | 1000 artikel
 bobot  atas bawah  #klu  sing%  cover%   silhou   davies    skor
  0.40  0.55  0.43   186    10%     90%    0.135    1.821   0.121
  0.40  0.60  0.48   281    15%     85%    0.214    1.569   0.181
  0.40  0.65  0.53   400    24%     76%    0.298    1.340   0.226
  0.40  0.70  0.58   492    34%     66%    0.390    1.186   0.258
  0.40  0.72  0.60   522    37%     63%    0.414    1.128   0.262
  0.40  0.75  0.63   565    41%     59%    0.446    1.055   0.264
  0.40  0.78  0.66   600    44%     55%    0.481    0.982   0.267
  0.40  0.80  0.68   627    47%     53%    0.522    0.914   0.275
  0.45  0.55  0.43   201    11%     89%    0.149    1.821   0.133
  0.45  0.60  0.48   289    16%     84%    0.219    1.603   0.184
  0.45  0.65  0.53   409    25%     75%    0.303    1.360   0.227
  0.45  0.70  0.58   502    35%     65%    0.393    1.203   0.257
  0.45  0.72  0.60   529    38%     62%    0.414

In [8]:
# ============================================================
# WORKER 2 — SUMMARIZATION (ROUGE + BERTScore)
# ============================================================
from concurrent.futures import ThreadPoolExecutor

from rouge_score import rouge_scorer

try:
    from bert_score import score as _bertscore
    _BERT_OK = True
except Exception:
    _BERT_OK = False
    print("[!] bert-score tidak tersedia — BERTScore akan dilewati.")

_scorer_rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

MAKS_PARALEL = 4  # jumlah request Groq bersamaan (rotasi key menjaga rate limit)


def summarize_artikel(teks):
    """Ringkas 1 artikel memakai PROMPT PRODUKSI Worker 2 (piramida terbalik + 5W1H, JSON),
    lalu ambil field 'rangkuman' untuk dibandingkan ke gold. Disamakan PERSIS dengan pipeline
    agar evaluasi mengukur prompt yang BENAR-BENAR dipakai produksi (bukan prompt lain)."""
    prompt = f"""Ringkas kumpulan potongan berita berikut (satu peristiwa) dan keluarkan JSON murni.

BERITA:
"{str(teks)[:2000]}"

Aturan:
1. judul: 1 kalimat gaya jurnalistik, inti seluruh peristiwa, 4-10 kata.
2. rangkuman: 4-15 kalimat (WAJIB minimal 4). Pakai struktur PIRAMIDA TERBALIK: kalimat-kalimat awal sudah memuat inti 5W1H (Apa, Siapa, Di mana, Kapan, Mengapa, Bagaimana) secara padat & jelas, detail pendukung menyusul makin ke bawah. Bila ada >1 peristiwa tak nyambung, pisahkan jadi beberapa paragraf.

Format: {{"judul":"...","rangkuman":"..."}}"""
    isi = groq_chat(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL_PEKERJA,
        temperature=0.2,
    )
    try:
        rangkuman = str(_ekstrak_json(isi).get("rangkuman", "")).strip()
        return rangkuman or str(isi).strip()
    except Exception:
        return str(isi).strip()


def _bertscore_f1(prediksi, referensi):
    """Rata-rata BERTScore-F1 (semantik, multilingual 'id'). None bila tak tersedia."""
    if not (_BERT_OK and prediksi):
        return None
    try:
        _, _, F = _bertscore(prediksi, referensi, lang="id", verbose=False)
        return round(float(F.mean()), 4)
    except Exception as e:
        print(f"    [!] BERTScore dilewati: {str(e)[:100]}")
        return None


def evaluasi_dataset_ringkasan(sampel, nama):
    """ROUGE-1/2/L + BERTScore untuk satu dataset summarization berlabel."""
    if not sampel:
        print(f"\n[{nama}] dilewati (dataset tidak tersedia).")
        return None

    print(f"\n[{nama}] meringkas {len(sampel)} artikel dengan {MODEL_PEKERJA} "
          f"(paralel x{MAKS_PARALEL})...")

    def _proses(row):
        try:
            return summarize_artikel(row["teks"]), row["ringkasan"]
        except Exception as e:
            print(f"  [!] sampel gagal: {str(e)[:100]}")
            return None

    with ThreadPoolExecutor(max_workers=MAKS_PARALEL) as ex:
        hasil_proses = list(ex.map(_proses, sampel))  # urutan tetap terjaga

    prediksi, referensi = [], []
    r1 = r2 = rl = 0.0
    for keluaran in hasil_proses:
        if keluaran is None:
            continue
        pred, ref = keluaran
        sc = _scorer_rouge.score(ref, pred)  # score(target, prediction)
        r1 += sc["rouge1"].fmeasure
        r2 += sc["rouge2"].fmeasure
        rl += sc["rougeL"].fmeasure
        prediksi.append(pred)
        referensi.append(ref)

    n = len(prediksi)
    if n == 0:
        print("  semua sampel gagal diproses.")
        return None

    hasil = {
        "rouge1": round(r1 / n, 4),
        "rouge2": round(r2 / n, 4),
        "rougeL": round(rl / n, 4),
        "bertscore": _bertscore_f1(prediksi, referensi),
        "n": n,
    }
    bs_txt = f" | BERTScore-F1 {hasil['bertscore']:.4f}" if hasil["bertscore"] is not None else ""
    print(f"  ROUGE-1 {hasil['rouge1']:.4f} | ROUGE-2 {hasil['rouge2']:.4f} | "
          f"ROUGE-L {hasil['rougeL']:.4f}{bs_txt}  (n={n})")
    return hasil


def evaluasi_lead_produksi():
    """Proxy: summary klaster produksi vs 2 kalimat awal artikel aslinya (ROUGE saja)."""
    res = supabase.table("tabel_cluster") \
        .select("id_cluster, summary_text") \
        .eq("status_summary", 1).limit(30).execute()

    r1 = r2 = 0.0
    n = 0
    for klaster in res.data:
        if not klaster["summary_text"]:
            continue
        res_art = supabase.table("tabel_berita") \
            .select("isi_teks").eq("id_cluster", klaster["id_cluster"]).limit(1).execute()
        if not res_art.data:
            continue
        isi = str(res_art.data[0]["isi_teks"])
        kalimat = pisah_kalimat(isi)
        lead = " ".join(kalimat[:2]) if len(kalimat) >= 2 else isi[:200]
        sc = _scorer_rouge.score(lead, klaster["summary_text"])
        r1 += sc["rouge1"].fmeasure
        r2 += sc["rouge2"].fmeasure
        n += 1

    if n == 0:
        return None
    return {"rouge1": round(r1 / n, 4), "rouge2": round(r2 / n, 4), "n": n}


def eval_summarization():
    print("\n" + "=" * 55)
    print("EVALUASI WORKER 2 — SUMMARIZATION (ROUGE + BERTScore)")
    print("=" * 55)
    hasil = {
        "xlsum": evaluasi_dataset_ringkasan(SAMPEL_XLSUM, "A. XL-Sum (referensi jurnalis BBC)"),
        "indosum": evaluasi_dataset_ringkasan(SAMPEL_INDOSUM, "B. IndoSum (domain berita mirip Briefly)"),
        "lead": evaluasi_lead_produksi(),
    }
    print("\nTarget: ROUGE-1 > 0.30 (ideal > 0.40) | ROUGE-2 > 0.12 | BERTScore-F1 > 0.70")
    print("Catatan: skor 'Lead' wajar paling rendah (proxy 2 kalimat vs ringkasan multi-artikel).")
    print("Catatan: prompt = PROMPT PRODUKSI Worker 2 (5W1H, 4-15 kalimat) -> ringkasan lebih panjang dari gold,")
    print("         jadi ROUGE leksikal wajar moderat; BERTScore (semantik) lebih adil menilai makna.")
    return hasil


HASIL["summarization"] = eval_summarization()


EVALUASI WORKER 2 — SUMMARIZATION (ROUGE + BERTScore)

[A. XL-Sum (referensi jurnalis BBC)] meringkas 200 artikel dengan llama-3.1-8b-instant (paralel x4)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ROUGE-1 0.1501 | ROUGE-2 0.0490 | ROUGE-L 0.1084 | BERTScore-F1 0.6839  (n=200)

[B. IndoSum (domain berita mirip Briefly)] meringkas 200 artikel dengan llama-3.1-8b-instant (paralel x4)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ROUGE-1 0.4115 | ROUGE-2 0.2582 | ROUGE-L 0.3108 | BERTScore-F1 0.7623  (n=200)

Target: ROUGE-1 > 0.30 (ideal > 0.40) | ROUGE-2 > 0.12 | BERTScore-F1 > 0.70
Catatan: skor 'Lead' wajar paling rendah (proxy 2 kalimat vs ringkasan multi-artikel).
Catatan: prompt = PROMPT PRODUKSI Worker 2 (5W1H, 4-15 kalimat) -> ringkasan lebih panjang dari gold,
         jadi ROUGE leksikal wajar moderat; BERTScore (semantik) lebih adil menilai makna.


In [9]:
# ============================================================
# WORKER 3 — SENTIMEN: Benchmark IndoNLU SmSA (label emas manusia)
#   Sentimen BINER: Positif / Negatif (Netral dihilangkan).
# ============================================================
from sklearn.metrics import classification_report


def klasifikasi_sentimen(teks):
    """Klasifikasi sentimen 1 teks -> Positif/Negatif (biner).
    Tanpa mode json_object (rawan 400 di model reasoning) -> JSON diekstrak manual."""
    prompt = (
        "Klasifikasikan sentimen keseluruhan teks berbahasa Indonesia berikut.\n"
        'Balas HANYA JSON: {"sentimen": "Positif"} atau {"sentimen": "Negatif"} '
        "(dua pilihan saja, tanpa Netral, tanpa penjelasan lain).\n\n"
        f'Teks: "{str(teks)[:1000]}"'
    )
    isi = groq_chat(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL_PEKERJA,
        temperature=0.0,
    )
    return _normalisasi_sentimen(_ekstrak_json(isi).get("sentimen"))


def eval_sentimen_smsa():
    print("\n" + "=" * 55)
    print("EVALUASI WORKER 3 — SENTIMEN vs IndoNLU SmSA (label emas, BINER)")
    print("=" * 55)

    if not SAMPEL_SMSA:
        print("Dataset SmSA tidak tersedia — dilewati.")
        return None

    print(f"Mengklasifikasi {len(SAMPEL_SMSA)} teks dengan {MODEL_PEKERJA} "
          f"(paralel x{MAKS_PARALEL})...")

    def _proses(row):
        try:
            return klasifikasi_sentimen(row["teks"]), row["emas"]
        except Exception as e:
            print(f"  [!] teks gagal: {str(e)[:100]}")
            return None

    with ThreadPoolExecutor(max_workers=MAKS_PARALEL) as ex:
        hasil_proses = list(ex.map(_proses, SAMPEL_SMSA))  # urutan terjaga

    y_pred = [h[0] for h in hasil_proses if h is not None]
    y_emas = [h[1] for h in hasil_proses if h is not None]

    if not y_emas:
        print("  semua teks gagal diproses.")
        return None

    label = ["Positif", "Negatif"]
    f1 = f1_score(y_emas, y_pred, average="macro", labels=label, zero_division=0)
    akurasi = sum(a == b for a, b in zip(y_emas, y_pred)) / len(y_emas)

    print(f"\nn = {len(y_emas)}  |  Akurasi = {akurasi:.4f}  |  F1-macro = {f1:.4f}  (target > 0.75)")
    print("\nLaporan per kelas:")
    print(classification_report(y_emas, y_pred, labels=label, zero_division=0))

    return {"f1_macro": round(float(f1), 4), "akurasi": round(float(akurasi), 4), "n": len(y_emas)}


HASIL["sentimen_smsa"] = eval_sentimen_smsa()


EVALUASI WORKER 3 — SENTIMEN vs IndoNLU SmSA (label emas, BINER)
Mengklasifikasi 350 teks dengan llama-3.1-8b-instant (paralel x4)...

n = 350  |  Akurasi = 0.9914  |  F1-macro = 0.9914  (target > 0.75)

Laporan per kelas:
              precision    recall  f1-score   support

     Positif       0.99      0.99      0.99       175
     Negatif       0.99      0.99      0.99       175

    accuracy                           0.99       350
   macro avg       0.99      0.99      0.99       350
weighted avg       0.99      0.99      0.99       350



In [10]:
# ============================================================
# WORKER 3 & 4 — LLM-as-Judge (model juri menilai output model pekerja)
#   Sentimen di sini = berbasis ASPEK/aktor pada ringkasan produksi (BINER: Positif/Negatif).
#   Dilengkapi PENGHITUNG kegagalan -> kalau N/A, ketahuan sebabnya (bukan diam-diam).
# ============================================================
# Penghitung global panggilan juri (di-reset tiap kali fungsi dijalankan).
_JUDGE_STAT = {"ok": 0, "gagal": 0, "err_terakhir": ""}


def llm_judge(prompt):
    """Kirim prompt ke model juri -> dict JSON, atau None bila gagal.
    Tanpa mode json_object (rawan 400 di model reasoning) -> JSON diekstrak manual.
    Retry + rotasi key ditangani groq_chat; kegagalan dicatat ke _JUDGE_STAT."""
    try:
        isi = groq_chat(
            messages=[{"role": "user", "content": prompt}],
            model=MODEL_JURI,
            temperature=0.1,
        )
        hasil = _ekstrak_json(isi)
        _JUDGE_STAT["ok"] += 1
        return hasil
    except Exception as e:
        _JUDGE_STAT["gagal"] += 1
        _JUDGE_STAT["err_terakhir"] = str(e)[:160]
        return None


def eval_sentimen_dan_prediksi(batas_klaster=10):
    print("\n" + "=" * 55)
    print("EVALUASI WORKER 3 & 4 — LLM-as-Judge (juri menilai pekerja)")
    print("=" * 55)
    _JUDGE_STAT.update({"ok": 0, "gagal": 0, "err_terakhir": ""})

    res = supabase.table("tabel_cluster") \
        .select("id_cluster, summary_text, judul_summary") \
        .eq("status_summary", 1) \
        .eq("status_prediksi", 1) \
        .limit(batas_klaster).execute()

    if not res.data:
        print("Tidak ada klaster dengan status_summary=1 & status_prediksi=1 — tidak ada yang dinilai.")
        return {"sentimen_f1": None, "sektor_acc": None, "risiko_acc": None}

    hasil_sentimen, hasil_prediksi = [], []

    for klaster in res.data:
        id_c = klaster["id_cluster"]
        ringkasan = klaster.get("summary_text") or ""
        if not ringkasan:
            continue
        print(f"  menilai klaster {id_c}...")

        # ── Sentimen aktor (BINER) ──
        res_aktor = supabase.table("tabel_sentimen_aktor") \
            .select("nama_aktor, sentimen").eq("id_cluster", id_c).execute()

        for aktor in res_aktor.data:
            prompt = f"""Kamu juri evaluasi analisis berita Indonesia. Balas HANYA JSON.

RINGKASAN:
{ringkasan[:500]}

AKTOR: {aktor['nama_aktor']}
SENTIMEN DIPREDIKSI: {aktor['sentimen']}

Apakah sentimen ini tepat berdasarkan ringkasan? Sentimen hanya Positif atau Negatif (tidak ada Netral). JSON:
{{"sentimen_tepat": true, "sentimen_yang_benar": "Positif/Negatif", "alasan": "1 kalimat"}}"""
            hasil = llm_judge(prompt)
            if hasil:
                hasil_sentimen.append({
                    "pred": _normalisasi_sentimen(aktor["sentimen"]),
                    "benar": _normalisasi_sentimen(hasil.get("sentimen_yang_benar")),
                    "tepat": bool(hasil.get("sentimen_tepat")),
                })

        # ── Prediksi sektor & risiko ──
        res_sektor = supabase.table("tabel_sektor") \
            .select("nama_sektor, tingkat_risiko").eq("id_cluster", id_c).execute()

        if res_sektor.data:
            sektor_list = [s["nama_sektor"] for s in res_sektor.data]
            risiko = res_sektor.data[0]["tingkat_risiko"]
            prompt2 = f"""Kamu analis kebijakan Indonesia yang mengevaluasi prediksi AI. Balas HANYA JSON.

RINGKASAN: {ringkasan[:500]}

SEKTOR DIPREDIKSI: {', '.join(sektor_list)}
RISIKO DIPREDIKSI: {risiko}

Nilai apakah prediksi ini tepat. JSON:
{{"sektor_tepat": true, "risiko_tepat": true, "risiko_yang_benar": "Tinggi/Sedang/Rendah", "alasan": "singkat"}}"""
            hasil2 = llm_judge(prompt2)
            if hasil2:
                hasil_prediksi.append(hasil2)

    hasil_akhir = {"sentimen_f1": None, "sektor_acc": None, "risiko_acc": None}

    # ── Metrik sentimen aspek (BINER) ──
    if hasil_sentimen:
        y_benar = [h["benar"] for h in hasil_sentimen]
        y_pred = [h["pred"] for h in hasil_sentimen]
        setuju = sum(h["tepat"] for h in hasil_sentimen)
        f1 = f1_score(y_benar, y_pred, average="macro",
                      labels=["Positif", "Negatif"], zero_division=0)
        hasil_akhir["sentimen_f1"] = round(float(f1), 4)
        print(f"\nSentimen aspek/aktor (dinilai juri) — n={len(hasil_sentimen)}")
        print(f"  Agreement rate : {setuju}/{len(hasil_sentimen)} = {setuju / len(hasil_sentimen):.1%}")
        print(f"  F1-macro       : {f1:.4f}")

    # ── Metrik prediksi ──
    if hasil_prediksi:
        total = len(hasil_prediksi)
        sektor_ok = sum(1 for h in hasil_prediksi if h.get("sektor_tepat"))
        risiko_ok = sum(1 for h in hasil_prediksi if h.get("risiko_tepat"))
        hasil_akhir["sektor_acc"] = round(sektor_ok / total, 4)
        hasil_akhir["risiko_acc"] = round(risiko_ok / total, 4)
        print(f"\nPrediksi dampak (dinilai juri) — n={total}")
        print(f"  Sektor tepat : {sektor_ok}/{total} = {sektor_ok / total:.1%}  (target > 70%)")
        print(f"  Risiko tepat : {risiko_ok}/{total} = {risiko_ok / total:.1%}  (target > 70%)")

    # ── Ringkasan kesehatan panggilan juri (agar N/A tidak misterius) ──
    total_call = _JUDGE_STAT["ok"] + _JUDGE_STAT["gagal"]
    print(f"\nPanggilan juri: {_JUDGE_STAT['ok']} OK / {_JUDGE_STAT['gagal']} GAGAL "
          f"(dari {total_call}).")
    if _JUDGE_STAT["gagal"]:
        print(f"  Contoh error terakhir: {_JUDGE_STAT['err_terakhir']}")
    if total_call and _JUDGE_STAT["ok"] == 0:
        print("  >> SEMUA panggilan juri gagal -> metrik N/A. Cek: model juri aktif? kuota Groq habis?")

    return hasil_akhir


HASIL["sentimen_prediksi"] = eval_sentimen_dan_prediksi(batas_klaster=10)


EVALUASI WORKER 3 & 4 — LLM-as-Judge (juri menilai pekerja)
  menilai klaster 2589...
  menilai klaster 5507...
  menilai klaster 2571...
  menilai klaster 2565...
  menilai klaster 2584...
  menilai klaster 2526...
  menilai klaster 5504...
  menilai klaster 5617...
  menilai klaster 4876...
  menilai klaster 4712...

Sentimen aspek/aktor (dinilai juri) — n=29
  Agreement rate : 15/29 = 51.7%
  F1-macro       : 0.7575

Prediksi dampak (dinilai juri) — n=10
  Sektor tepat : 8/10 = 80.0%  (target > 70%)
  Risiko tepat : 7/10 = 70.0%  (target > 70%)

Panggilan juri: 39 OK / 0 GAGAL (dari 39).


In [11]:
# ============================================================
# WORKER 4 — GROUND TRUTH MANUSIA (pembanding NON-LLM untuk prediksi)
#   LLM-as-judge bisa bias (model menilai keluarga model sendiri). Di sini prediksi
#   sektor & risiko dibandingkan ke LABEL MANUSIA (gold) -> pembanding yang kredibel.
#   Alur sekali-isi:
#     1) Jika file gold belum ada -> tulis TEMPLATE (ringkasan + prediksi AI).
#        Buka CSV-nya, isi kolom 'sektor_benar' (pisah ';') & 'risiko_benar', simpan.
#     2) Jalankan ulang sel ini -> akurasi AI vs gold manusia dihitung.
# ============================================================
import os
import csv
import re

GOLD_PREDIKSI_PATH = r"gold_prediksi_valid.csv"
SEKTOR_SEP = ";"   # pemisah multi-sektor di kolom gold


def _norm_sektor(x):
    return str(x).strip().capitalize()


def _norm_risiko(x):
    x = str(x).strip().capitalize()
    return x if x in ("Tinggi", "Sedang", "Rendah") else ""


def _prediksi_ai_per_klaster(id_list):
    """Kumpulkan sektor (set) & risiko hasil AI untuk daftar id_cluster (1 query)."""
    ids_int = [int(x) for x in id_list if str(x).isdigit()]
    if not ids_int:
        return {}
    rs = (supabase.table("tabel_sektor")
          .select("id_cluster, nama_sektor, tingkat_risiko")
          .in_("id_cluster", ids_int).execute())
    pred = {}
    for s in rs.data:
        idc = str(s["id_cluster"])
        pred.setdefault(idc, {"sektor": set(), "risiko": ""})
        pred[idc]["sektor"].add(_norm_sektor(s["nama_sektor"]))
        if not pred[idc]["risiko"]:
            pred[idc]["risiko"] = _norm_risiko(s["tingkat_risiko"])
    return pred


def _tulis_template_gold(batas=40):
    """Tulis CSV berisi klaster terprediksi + prediksi AI; kolom kebenaran dikosongkan."""
    res = (supabase.table("tabel_cluster")
           .select("id_cluster, judul_summary, summary_text")
           .eq("status_prediksi", 1).eq("status_summary", 1)
           .limit(batas).execute())
    if not res.data:
        print("Tidak ada klaster terprediksi untuk dilabeli. Jalankan Worker 4 dulu.")
        return
    pred = _prediksi_ai_per_klaster([k["id_cluster"] for k in res.data])
    with open(GOLD_PREDIKSI_PATH, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["id_cluster", "judul", "ringkasan", "sektor_ai", "risiko_ai",
                    "sektor_benar", "risiko_benar"])
        for k in res.data:
            idc = str(k["id_cluster"])
            p = pred.get(idc, {"sektor": set(), "risiko": ""})
            w.writerow([idc, k.get("judul_summary", ""),
                        (k.get("summary_text", "") or "")[:300],
                        SEKTOR_SEP.join(sorted(p["sektor"])), p["risiko"], "", ""])
    print(f"Template gold ditulis: {GOLD_PREDIKSI_PATH}  ({len(res.data)} klaster)")
    print("LANGKAH: buka CSV itu -> isi 'sektor_benar' (boleh banyak, pisah ';') &")
    print("'risiko_benar' (Tinggi/Sedang/Rendah) untuk minimal ~20 baris -> simpan ->")
    print("jalankan ULANG sel ini untuk menghitung akurasi vs gold manusia.")


def _baca_gold(path):
    """Baca gold label manusia secara ROBUS terhadap kerusakan Excel.

    Excel locale Indonesia memakai ';' sebagai pemisah daftar, sehingga saat file
    template (dipisah koma) dibuka & disimpan ulang: header jadi 'risiko_benar;;;;'
    dan sebagian baris (yang ringkasannya mengandung koma) ter-quote aneh -> parser
    csv biasa gagal (id_cluster jadi sampah & kolom 'risiko_benar' tak terbaca).

    Strategi tahan-banting: id_cluster = angka di AWAL baris; dua kolom-koma TERAKHIR
    = sektor_benar (multi-nilai dipisah ';') & risiko_benar. Berlaku utk baris bersih
    maupun yang rusak, karena ekor baris selalu '...,<sektor_benar>,<risiko_benar>;'.
    """
    gold = {}
    with open(path, encoding="utf-8-sig") as f:
        baris = f.read().splitlines()
    for ln in baris[1:]:                         # lewati header
        s = ln.strip()
        if not s:
            continue
        m = re.match(r'"?(\d+)', s)             # id_cluster di awal (boleh diawali kutip)
        if not m:
            continue
        tok = s.rstrip(";").rstrip().split(",")  # buang ';' sampah di ujung lalu pisah koma
        if len(tok) < 2:
            continue
        sb = {_norm_sektor(x) for x in tok[-2].split(SEKTOR_SEP) if x.strip()}
        rb = _norm_risiko(tok[-1])
        if sb or rb:
            gold[m.group(1)] = {"sektor": sb, "risiko": rb}
    return gold


def eval_prediksi_vs_gold():
    print("\n" + "=" * 55)
    print("EVALUASI WORKER 4 — PREDIKSI vs GROUND TRUTH MANUSIA")
    print("=" * 55)

    if not os.path.exists(GOLD_PREDIKSI_PATH):
        print("File gold belum ada -> membuat template untuk kamu isi...")
        _tulis_template_gold()
        return None

    # Baca baris gold yang SUDAH diisi (robus thd kerusakan Excel ';').
    gold = _baca_gold(GOLD_PREDIKSI_PATH)

    if not gold:
        print(f"Belum ada baris terisi di {GOLD_PREDIKSI_PATH}.")
        print("Isi kolom 'sektor_benar'/'risiko_benar' dulu, lalu jalankan ulang.")
        return None

    pred = _prediksi_ai_per_klaster(list(gold.keys()))
    sektor_hit = sektor_exact = risiko_hit = n_sektor = n_risiko = 0
    for idc, g in gold.items():
        p = pred.get(idc, {"sektor": set(), "risiko": ""})
        if g["sektor"]:
            n_sektor += 1
            if p["sektor"] & g["sektor"]:
                sektor_hit += 1
            if p["sektor"] == g["sektor"]:
                sektor_exact += 1
        if g["risiko"]:
            n_risiko += 1
            if p["risiko"] == g["risiko"]:
                risiko_hit += 1

    hasil = {
        "sektor_acc_lenient": round(sektor_hit / n_sektor, 4) if n_sektor else None,
        "sektor_acc_exact": round(sektor_exact / n_sektor, 4) if n_sektor else None,
        "risiko_acc": round(risiko_hit / n_risiko, 4) if n_risiko else None,
        "n_sektor": n_sektor, "n_risiko": n_risiko,
    }
    print(f"Dinilai vs LABEL MANUSIA (n_sektor={n_sektor}, n_risiko={n_risiko}):")
    if n_sektor:
        print(f"  Sektor tepat (>=1 benar) : {sektor_hit}/{n_sektor} = {hasil['sektor_acc_lenient']:.1%}")
        print(f"  Sektor sama persis        : {sektor_exact}/{n_sektor} = {hasil['sektor_acc_exact']:.1%}")
    if n_risiko:
        print(f"  Risiko tepat (exact)      : {risiko_hit}/{n_risiko} = {hasil['risiko_acc']:.1%}")
    print("\nIni pembanding NON-LLM (label manusia) — lebih kredibel daripada LLM-as-judge.")
    return hasil


HASIL["prediksi_gold"] = eval_prediksi_vs_gold()


EVALUASI WORKER 4 — PREDIKSI vs GROUND TRUTH MANUSIA
Dinilai vs LABEL MANUSIA (n_sektor=40, n_risiko=40):
  Sektor tepat (>=1 benar) : 17/40 = 42.5%
  Sektor sama persis        : 0/40 = 0.0%
  Risiko tepat (exact)      : 9/40 = 22.5%

Ini pembanding NON-LLM (label manusia) — lebih kredibel daripada LLM-as-judge.


In [12]:
# ============================================================
# LAPORAN AKHIR
# ============================================================
def _fmt(nilai, spec="", default="N/A"):
    """Format nilai dengan aman; kembalikan default bila None/tak bisa diformat."""
    if nilai is None:
        return default
    try:
        return format(nilai, spec) if spec else str(nilai)
    except (ValueError, TypeError):
        return str(nilai)


def cetak_laporan():
    print("\n" + "=" * 60)
    print("           LAPORAN EVALUASI SISTEM BRIEFLY")
    print("=" * 60)

    c = HASIL.get("clustering") or {}
    s = HASIL.get("summarization") or {}
    smsa = HASIL.get("sentimen_smsa") or {}
    sp = HASIL.get("sentimen_prediksi") or {}
    g = HASIL.get("prediksi_gold") or {}
    xlsum = s.get("xlsum") or {}
    indosum = s.get("indosum") or {}
    lead = s.get("lead") or {}

    print("\n[WORKER 1] CLUSTERING")
    print(f"  Silhouette Score   : {_fmt(c.get('silhouette'))}  (target > 0.25)")
    print(f"  Davies-Bouldin     : {_fmt(c.get('davies_bouldin'))}  (target < 1.0)")

    print("\n[WORKER 2] SUMMARIZATION   (ROUGE-1 / ROUGE-2 / BERTScore-F1)")
    print(f"  XL-Sum  (BBC)      : {_fmt(xlsum.get('rouge1'), '.4f')} / "
          f"{_fmt(xlsum.get('rouge2'), '.4f')} / {_fmt(xlsum.get('bertscore'), '.4f')}")
    print(f"  IndoSum (berita)   : {_fmt(indosum.get('rouge1'), '.4f')} / "
          f"{_fmt(indosum.get('rouge2'), '.4f')} / {_fmt(indosum.get('bertscore'), '.4f')}")
    print(f"  Lead (proxy prod.) : {_fmt(lead.get('rouge1'), '.4f')} / "
          f"{_fmt(lead.get('rouge2'), '.4f')} / -")

    print("\n[WORKER 3] SENTIMEN")
    print(f"  SmSA F1-macro      : {_fmt(smsa.get('f1_macro'))}  (label emas, target > 0.75)")
    print(f"  SmSA Akurasi       : {_fmt(smsa.get('akurasi'), '.1%')}")
    print(f"  Aspek/aktor (judge): F1 {_fmt(sp.get('sentimen_f1'))}  (LLM-as-Judge, sekunder)")

    print("\n[WORKER 4] PREDIKSI DAMPAK")
    print("  - LLM-as-Judge (sekunder, bisa bias):")
    print(f"      Akurasi Sektor : {_fmt(sp.get('sektor_acc'), '.1%')}  (target > 70%)")
    print(f"      Akurasi Risiko : {_fmt(sp.get('risiko_acc'), '.1%')}")
    print("  - vs Ground Truth manusia (UTAMA, non-LLM):")
    print(f"      Sektor >=1 benar : {_fmt(g.get('sektor_acc_lenient'), '.1%')}  (n={g.get('n_sektor', 0)})")
    print(f"      Sektor exact     : {_fmt(g.get('sektor_acc_exact'), '.1%')}")
    print(f"      Risiko exact     : {_fmt(g.get('risiko_acc'), '.1%')}  (n={g.get('n_risiko', 0)})")

    print("\n" + "=" * 60)
    print("SUMBER GROUND TRUTH:")
    print("  Clustering : metrik internal tanpa label (unsupervised)")
    print("  Summary    : XL-Sum (BBC) + IndoSum (Kurniawan & Louvan) — ROUGE & BERTScore")
    print("  Sentimen   : IndoNLU SmSA (label emas manusia) + LLM-as-Judge (aspek)")
    print("  Prediksi   : LABEL MANUSIA (gold_prediksi.csv) + LLM-as-Judge (sekunder)")
    print("=" * 60)


cetak_laporan()


           LAPORAN EVALUASI SISTEM BRIEFLY

[WORKER 1] CLUSTERING
  Silhouette Score   : 0.288  (target > 0.25)
  Davies-Bouldin     : 1.7749  (target < 1.0)

[WORKER 2] SUMMARIZATION   (ROUGE-1 / ROUGE-2 / BERTScore-F1)
  XL-Sum  (BBC)      : 0.1501 / 0.0490 / 0.6839
  IndoSum (berita)   : 0.4115 / 0.2582 / 0.7623
  Lead (proxy prod.) : 0.3247 / 0.1855 / -

[WORKER 3] SENTIMEN
  SmSA F1-macro      : 0.9914  (label emas, target > 0.75)
  SmSA Akurasi       : 99.1%
  Aspek/aktor (judge): F1 0.7575  (LLM-as-Judge, sekunder)

[WORKER 4] PREDIKSI DAMPAK
  - LLM-as-Judge (sekunder, bisa bias):
      Akurasi Sektor : 80.0%  (target > 70%)
      Akurasi Risiko : 70.0%
  - vs Ground Truth manusia (UTAMA, non-LLM):
      Sektor >=1 benar : 42.5%  (n=40)
      Sektor exact     : 0.0%
      Risiko exact     : 22.5%  (n=40)

SUMBER GROUND TRUTH:
  Clustering : metrik internal tanpa label (unsupervised)
  Summary    : XL-Sum (BBC) + IndoSum (Kurniawan & Louvan) — ROUGE & BERTScore
  Sentimen   : I